# Validación Temporal — Dengue SIN Signos de Alarma (DSSA)
## Contraste con DCSA: sin ruptura estructural, con drift real

---

## Motivación

El notebook `03b` reveló una ruptura estructural masiva en DCSA:
la tasa de hospitalización saltó de 40% a 87% en 2022, causada por
el cambio de protocolo INS 2021. El modelo se degradó completamente.

En DSSA el panorama es **fundamentalmente diferente**:

| Aspecto | DCSA (03b) | DSSA (03d) |
|---------|-----------|----------|
| Tasa hosp. train | 40.0% | **3.7%** |
| Tasa hosp. test | 87.0% | **4.0%** |
| Cambio entre períodos | +47 pp | **+0.3 pp** |
| Ruptura estructural | Sí — masiva | **No detectada** |
| Tipo de drift | Metodológico (INS) | **Clínico real** |

En DSSA no hay ruptura estructural: la prevalencia es estable entre períodos.
Sin embargo, existe drift real en algunas manifestaciones clínicas —
particularmente ERUPCIONR y ARTRALGIA — posiblemente asociado a cambios
en los serotipos circulantes de dengue entre períodos.

La pregunta de validación temporal en DSSA es diferente:
> **Sin ruptura estructural, ¿cuánto se degrada el modelo al aplicarse
> sobre un período dominado por el año 2024 (49% de los casos)?**

**Anterior:** `03c_modelado_sin_signos.ipynb`

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# BANDERA DE EJECUCION
# FORZAR_RECALCULO = True  → recalcula todo desde cero
# FORZAR_RECALCULO = False → reutiliza checkpoints
# ═══════════════════════════════════════════════════════════════════════
FORZAR_RECALCULO = False

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
import joblib
import gc
from pathlib import Path
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.metrics import (
    f1_score, roc_auc_score, balanced_accuracy_score,
    precision_score, recall_score, average_precision_score,
    brier_score_loss, confusion_matrix,
    classification_report, make_scorer, RocCurveDisplay, PrecisionRecallDisplay)

warnings.filterwarnings('ignore')
%matplotlib inline

plt.rcParams.update({
    'figure.dpi': 150, 'savefig.dpi': 300,
    'figure.facecolor': 'white', 'axes.facecolor': 'white',
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.titlesize': 13, 'axes.titleweight': 'bold',
    'axes.labelsize': 11, 'legend.frameon': False,
})
COLORS = {
    'primary': '#2C7BB6', 'secondary': '#D7191C',
    'accent': '#FDAE61', 'neutral': '#7F7F7F',
    'train': '#2C7BB6', 'test': '#D7191C',
}
SIGNOS_ALARMA      = ['DOLOR_ABDO','VOMITO','DIARREA','SOMNOLENCI','HIPOTENSIO',
                      'HEPATOMEG','HEM_MUCOSA','HIPOTERMIA','AUM_HEMATO','CAIDA_PLAQ','ACUM_LIQUI']
SINTOMAS_GENERALES = ['FIEBRE','CEFALEA','DOLRRETROO','MALGIAS','ARTRALGIA','ERUPCIONR']
TARGET             = 'TARGET_HOSPITALIZACION_ORIGINAL'
RANDOM_STATE       = 42

ROOT        = Path(r'D:\Documentos\GitHub\dengue')
OUT_FIG     = ROOT / 'outputs' / 'figures'
OUT_TAB     = ROOT / 'outputs' / 'tables'
INTERIM     = ROOT / 'data' / 'interim'
CHECKPOINTS = ROOT / 'outputs' / 'checkpoints'
for p in [OUT_FIG, OUT_TAB, INTERIM, CHECKPOINTS]:
    p.mkdir(parents=True, exist_ok=True)

def checkpoint_existe(path):
    if FORZAR_RECALCULO: return False
    return Path(path).exists()

def guardar_checkpoint(obj, path, nombre=''):
    joblib.dump(obj, path)
    mb = Path(path).stat().st_size / 1e6
    print(f'  Guardado {nombre}: {Path(path).name} ({mb:.1f} MB)')

def limpiar_memoria(*objetos):
    for obj in objetos:
        try: del obj
        except: pass
    gc.collect()

def memoria_usada():
    import psutil, os
    mb = psutil.Process(os.getpid()).memory_info().rss / 1e6
    print(f'  RAM en uso: {mb:.0f} MB')

def metricas_extendidas_dssa(y_true, y_pred, y_prob, escenario):
    """Panel completo congruente con 03c. Métrica primaria: AUC-PR."""
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred).ravel()
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0
    return {
        'Escenario':   escenario,
        'AUC-PR':      round(average_precision_score(y_true, y_prob), 4),
        'AUC-ROC':     round(roc_auc_score(y_true, y_prob), 4),
        'F1':          round(f1_score(y_true, y_pred), 4),
        'Brier':       round(brier_score_loss(y_true, y_prob), 4),
        'Bal_Acc':     round(balanced_accuracy_score(y_true, y_pred), 4),
        'Precision':   round(precision_score(y_true, y_pred, zero_division=0), 4),
        'Recall':      round(recall_score(y_true, y_pred), 4),
        'Specificity': round(spec, 4),
    }

df_sin = joblib.load(INTERIM / 'datos_sin_signos.pkl')
print(f'DSSA total: {len(df_sin):,} casos')
print(f'Rango temporal: {df_sin["AÑO"].min()} – {df_sin["AÑO"].max()}')

## 1. Análisis del drift temporal — DSSA

A diferencia del DCSA, el análisis de distribución temporal en DSSA
revela un panorama muy diferente. Se cuantifican:
la estabilidad de la tasa de hospitalización entre períodos
y el drift en la prevalencia de síntomas.

### Hallazgo previo al análisis

El análisis preliminar de los datos revela características únicas del DSSA:

- **2017–2020 prácticamente vacíos:** solo 5 casos en esos 4 años — no hay
  datos representativos del período intermedio para entrenar el modelo.
- **2024 domina el test set:** concentra aproximadamente la mitad de todos
  los casos DSSA — el modelo temporal se evalúa principalmente sobre
  este año, lo que puede sesgar los resultados.
- **Sin ruptura estructural:** la tasa de hospitalización es estable entre
  períodos (~3.7% vs ~4.0%) — el cambio de protocolo INS 2021 que devastó
  al DCSA no tiene el mismo efecto en DSSA.

In [ ]:
# Análisis temporal DSSA
ANIO_CORTE = 2022

df_train_t = df_sin[df_sin['AÑO'] < ANIO_CORTE].copy()
df_test_t  = df_sin[df_sin['AÑO'] >= ANIO_CORTE].copy()

print('Distribución por período:')
print(f'  Train temporal (2015-2021): {len(df_train_t):,} casos | '
      f'Hosp: {df_train_t[TARGET].mean():.1%}')
print(f'  Test temporal  (2022-2025): {len(df_test_t):,} casos  | '
      f'Hosp: {df_test_t[TARGET].mean():.1%}')

anio_dist = df_sin.groupby('AÑO').agg(
    N=(TARGET,'size'), Hosp=(TARGET, lambda x: x.eq(1).sum())
).reset_index()
anio_dist['Pct_hosp'] = (anio_dist['Hosp']/anio_dist['N']*100).round(1)
anio_dist['Periodo']  = anio_dist['AÑO'].apply(
    lambda x: 'Train (2015-2021)' if x < ANIO_CORTE else 'Test (2022-2025)')
display(anio_dist)

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel A: volumen por año
colores_n = [COLORS['train'] if a < ANIO_CORTE else COLORS['test']
    for a in anio_dist['AÑO']]
bars_n = axes[0].bar(anio_dist['AÑO'].astype(str), anio_dist['N'],
    color=colores_n, alpha=0.85, edgecolor='white', width=0.7)
for bar, n in zip(bars_n, anio_dist['N']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 30,
        f'{n:,}', ha='center', va='bottom', fontsize=8)
axes[0].axvline(4.5, color='black', linewidth=2, linestyle='--')
axes[0].set_title('A. Volumen de casos DSSA por año\n'
    '⚠ 2017-2020 vacíos · 2024 domina el test set')
axes[0].set_xlabel('Año')
axes[0].set_ylabel('Número de casos')
axes[0].grid(axis='y', alpha=0.3)

# Panel B: tasa de hospitalización
colores_h = [COLORS['train'] if a < ANIO_CORTE else COLORS['test']
    for a in anio_dist['AÑO']]
bars_h = axes[1].bar(anio_dist['AÑO'].astype(str), anio_dist['Pct_hosp'],
    color=colores_h, alpha=0.85, edgecolor='white', width=0.7)
for bar, pct in zip(bars_h, anio_dist['Pct_hosp']):
    if pct > 0:
        axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{pct:.1f}%', ha='center', va='bottom', fontsize=8)
axes[1].axvline(4.5, color='black', linewidth=2, linestyle='--')
axes[1].axhline(df_sin[TARGET].mean()*100, color=COLORS['neutral'],
    linestyle=':', linewidth=1.5, label=f'Prevalencia global: {df_sin[TARGET].mean():.1%}')
axes[1].set_title('B. Tasa de hospitalización DSSA por año\n'
    'Sin ruptura estructural — contraste con DCSA (+47pp)')
axes[1].set_xlabel('Año')
axes[1].set_ylabel('% hospitalizados')
axes[1].legend(fontsize=9)
axes[1].grid(axis='y', alpha=0.3)

leyenda = [mpatches.Patch(color=COLORS['train'], label='Train 2015-2021'),
           mpatches.Patch(color=COLORS['test'],  label='Test 2022-2025')]
fig.legend(handles=leyenda, loc='lower center', ncol=2, fontsize=10,
    bbox_to_anchor=(0.5, -0.05))
plt.suptitle('Distribución temporal DSSA — Sin ruptura estructural',
    fontsize=13, fontweight='bold')
plt.tight_layout(rect=[0, 0.05, 1, 1])
fig.savefig(OUT_FIG / 'temporal_dssa_distribucion.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

In [ ]:
# Drift en manifestaciones clínicas DSSA
# En DSSA el drift es real pero inverso al DCSA:
# algunas variables BAJAN en prevalencia en el período reciente
todas = SINTOMAS_GENERALES + SIGNOS_ALARMA

prev_train = (df_train_t[todas].eq(1).mean()*100).rename('Train 2015-2021')
prev_test  = (df_test_t[todas].eq(1).mean()*100).rename('Test 2022-2025')
drift_df   = pd.concat([prev_train, prev_test], axis=1)
drift_df['Δpp']      = (drift_df['Test 2022-2025'] - drift_df['Train 2015-2021']).round(1)
drift_df['Drift_abs'] = drift_df['Δpp'].abs()
drift_df = drift_df.sort_values('Δpp', ascending=True)  # mayor caída primero

print('Drift en prevalencia de manifestaciones clínicas — DSSA:')
display(drift_df.round(1))
drift_df.to_csv(OUT_TAB / 'drift_prevalencia_dssa.csv')

# Gráfico — solo variables con drift > 2pp para no saturar
drift_relevante = drift_df[drift_df['Drift_abs'] > 2].copy()
fig, ax = plt.subplots(figsize=(12, 6))
x = np.arange(len(drift_relevante))
w = 0.35
ax.bar(x - w/2, drift_relevante['Train 2015-2021'], w,
    color=COLORS['train'], alpha=0.85, label='Train 2015-2021', edgecolor='white')
ax.bar(x + w/2, drift_relevante['Test 2022-2025'], w,
    color=COLORS['test'], alpha=0.85, label='Test 2022-2025', edgecolor='white')
for xi, (_, row) in enumerate(drift_relevante.iterrows()):
    delta = row['Δpp']
    y_max = max(row['Train 2015-2021'], row['Test 2022-2025']) + 1.5
    ax.text(xi, y_max, f'{delta:+.0f}pp',
        ha='center', fontsize=8.5, fontweight='bold',
        color=COLORS['test'] if delta > 0 else COLORS['train'])
ax.set_xticks(x)
ax.set_xticklabels(drift_relevante.index, rotation=35, ha='right', fontsize=9)
ax.set_ylabel('Prevalencia (%)')
ax.set_title(
    'Drift en manifestaciones clínicas DSSA (variables con |Δ| > 2pp)\n'
    'Drift inverso al DCSA: algunas variables BAJAN en el período reciente',
    fontsize=12, fontweight='bold')
ax.legend(fontsize=10)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIG / 'temporal_dssa_drift.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

### Interpretación del drift en DSSA

El patrón de drift en DSSA es **opuesto al DCSA** y epidemiológicamente
más coherente:

**ERUPCIONR cae significativamente entre períodos.**
La erupción cutánea es característica de ciertos serotipos del virus dengue
y su prevalencia puede variar entre brotes según el serotipo dominante.
Una caída real en ERUPCIONR sugiere un cambio en la composición
del serotipo circulante — drift epidemiológico, no metodológico.

**ARTRALGIA y DOLRRETROO también disminuyen.**
Estas manifestaciones musculoesqueléticas también varían por serotipo.
Su caída conjunta con ERUPCIONR refuerza la hipótesis de cambio
en el serotipo dominante entre 2015–2021 y 2022–2025.

**Contraste con DCSA:** en DCSA el drift fue metodológico (cambio en el
llenado del formulario INS — FIEBRE subió de 21% a 100%).
En DSSA el drift es epidemiológico real — las variables que cambian
son clínicamente coherentes con variaciones de serotipo.

**Implicación para el modelo:** el drift epidemiológico en DSSA
es menos perjudicial para el modelo que el drift metodológico en DCSA.
Sin embargo, si el modelo aprendió a usar ERUPCIONR o ARTRALGIA
como predictores, su importancia habrá cambiado entre períodos.

## 2. Preparación de features y evaluación temporal

Se cargan el modelo de 03c y los mismos 24 features sin AÑO.
La partición temporal usa el mismo corte 2022 que en DCSA
para mantener consistencia metodológica entre estratos.

**Diferencia crítica respecto a 03b:** el train temporal DSSA
tiene solo ~3,947 casos (vs ~6,000 en DCSA) y los años
2017–2020 están prácticamente vacíos — el modelo se entrena
principalmente con datos de 2015–2016, 2021 y el brote de 2019.

In [ ]:
# Cargar mejor modelo de 03c
CP_MODELO_03C = CHECKPOINTS / 'mejor_modelo_DSSA.pkl'
assert CP_MODELO_03C.exists(), 'Ejecuta primero 03c_modelado_sin_signos.ipynb'

saved_03c  = joblib.load(CP_MODELO_03C)
modelo_03c = saved_03c['pipeline']
mejor_03c  = saved_03c['modelo']
FEATURES   = saved_03c['features']

assert 'AÑO' not in FEATURES, 'ERROR: AÑO no debe estar en FEATURES'

print(f'Modelo cargado: {mejor_03c}')
print(f'Features ({len(FEATURES)}): {FEATURES}')
aucp_03c = saved_03c['tabla_test'].loc[mejor_03c, 'AUC-PR']
print(f'AUC-PR en test aleatorio (03c): {aucp_03c:.4f}')

vars_clinicas = SINTOMAS_GENERALES + SIGNOS_ALARMA
vars_demo     = [v for v in FEATURES if v not in vars_clinicas + ['SEMANA']]

def preparar_X(df):
    df_m = df[FEATURES + [TARGET]].copy()
    for v in vars_clinicas:
        df_m[v] = df_m[v].map({1: 1, 2: 0})
    for v in vars_demo:
        if str(df_m[v].dtype) in ('object',) or str(df_m[v].dtype).startswith('string'):
            df_m[v] = pd.Categorical(df_m[v]).codes.astype(float)
            df_m[v] = df_m[v].replace(-1, np.nan)
    for col in FEATURES:
        df_m[col] = pd.to_numeric(df_m[col], errors='coerce').astype(np.float64)
    return df_m.dropna(subset=[TARGET])

df_train_prep = preparar_X(df_train_t)
df_test_prep  = preparar_X(df_test_t)

X_train_t = df_train_prep[FEATURES]
y_train_t = df_train_prep[TARGET].astype(int)
X_test_t  = df_test_prep[FEATURES]
y_test_t  = df_test_prep[TARGET].astype(int)

print(f'\nTrain temporal: {len(X_train_t):,} | Hosp: {y_train_t.mean():.1%}')
print(f'Test temporal : {len(X_test_t):,}  | Hosp: {y_test_t.mean():.1%}')
print(f'\nPositivos en train: {y_train_t.sum()} | en test: {y_test_t.sum()}')

## 3. Evaluación del modelo 03c sobre el test temporal (Escenario B)

A diferencia del DCSA donde el escenario B producía un artefacto de F1 alto
por el cambio de distribución del target (40%→87%), en DSSA la distribución
es estable (~3.7%→4.0%). Por tanto, cualquier cambio en las métricas
refleja **degradación o mejora real del modelo**, no un artefacto distribucional.

Se incluye la partición aleatoria de 03c como referencia (escenario A).

In [ ]:
# Cargar partición aleatoria de 03c para escenario A
CP_PART = CHECKPOINTS / 'particion_DSSA.pkl'
if CP_PART.exists():
    _, X_test_al, _, y_test_al, _ = joblib.load(CP_PART)
    tiene_aleatorio = True
else:
    tiene_aleatorio = False
    print('Partición aleatoria no disponible')

# Escenario B: modelo 03c sobre test temporal
y_prob_03c = modelo_03c.predict_proba(X_test_t)[:, 1]
y_pred_03c = modelo_03c.predict(X_test_t)

m_B = metricas_extendidas_dssa(y_test_t, y_pred_03c, y_prob_03c,
    'B. Modelo 03c → test temporal (2022-2025)')

if tiene_aleatorio:
    y_pred_al = modelo_03c.predict(X_test_al)
    y_prob_al = modelo_03c.predict_proba(X_test_al)[:, 1]
    m_A = metricas_extendidas_dssa(y_test_al, y_pred_al, y_prob_al,
        'A. Test aleatorio (03c)')
else:
    m_A = {
        'Escenario': 'A. Test aleatorio (03c)',
        'AUC-PR': aucp_03c,
    }

tabla_AB = pd.DataFrame([m_A, m_B])
print('Comparativa escenario A vs B:')
display(tabla_AB.set_index('Escenario'))

print(f'\nCambio de métricas: Test aleatorio (A) → Test temporal (B)')
for metrica in ['AUC-PR', 'AUC-ROC', 'Bal_Acc', 'Recall', 'Specificity', 'Brier']:
    if metrica in m_A and metrica in m_B:
        delta = m_B[metrica] - m_A[metrica]
        if metrica == 'Brier':
            interp = '↓ mejora calibración' if delta < 0 else '↑ peor calibración'
        else:
            interp = '↑ mejora' if delta > 0 else '↓ degradación'
        print(f'  Δ{metrica:<15}: {delta:+.4f}  {interp}')

print(f'\nReporte de clasificación — test temporal:')
print(classification_report(y_test_t, y_pred_03c,
    target_names=['No hosp.', 'Hospitalizado']))

## 4. Re-entrenamiento con partición temporal (Escenario C)

Se re-entrena el mismo pipeline sobre datos 2015–2021 exclusivamente.

**Advertencia sobre el train temporal DSSA:**
El train tiene solo ~146 positivos (hospitalizados) en 3,947 casos.
Los años 2017–2020 tienen 5 casos en total — el modelo aprende
casi exclusivamente de 2015–2016, 2019 y 2021.
Con tan pocos positivos, el re-entrenamiento es más ruidoso que en DCSA
y los resultados pueden variar significativamente entre ejecuciones.

In [ ]:
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from imblearn.over_sampling import SMOTE
from imblearn.pipeline import Pipeline as ImbPipeline

CP_TEMPORAL = CHECKPOINTS / 'eval_temporal_DSSA.pkl'

if checkpoint_existe(CP_TEMPORAL):
    print('Re-entrenamiento temporal ya calculado — cargando...')
    saved_temp = joblib.load(CP_TEMPORAL)
    y_pred_temp   = saved_temp['y_pred']
    y_prob_temp   = saved_temp['y_prob']
    metricas_temp = saved_temp['metricas']
else:
    vars_clinicas_l = SINTOMAS_GENERALES + SIGNOS_ALARMA
    vars_demo_l     = [v for v in FEATURES if v not in vars_clinicas_l + ['SEMANA']]
    vars_tiempo_l   = [v for v in FEATURES if v in ['SEMANA']]
    cols_num        = vars_demo_l + vars_tiempo_l

    prep_temp = ColumnTransformer([
        ('num', Pipeline([
            ('imp', SimpleImputer(strategy='median')),
            ('sc',  StandardScaler())
        ]), cols_num),
        ('bin', SimpleImputer(strategy='most_frequent'), vars_clinicas_l),
    ], remainder='drop')

    # k=3 — consistente con 03c (pocos positivos en DSSA)
    pipe_temp = ImbPipeline([
        ('prep',  prep_temp),
        ('smote', SMOTE(random_state=RANDOM_STATE, k_neighbors=3)),
        ('clf',   ExtraTreesClassifier(n_estimators=300, max_depth=10,
                      random_state=RANDOM_STATE, n_jobs=1)),
    ])

    print(f'Entrenando {mejor_03c} con datos 2015-2021...')
    print(f'  Positivos en train: {y_train_t.sum()} de {len(y_train_t):,} casos')
    pipe_temp.fit(X_train_t, y_train_t)
    y_prob_temp = pipe_temp.predict_proba(X_test_t)[:, 1]
    y_pred_temp = pipe_temp.predict(X_test_t)

    metricas_temp = metricas_extendidas_dssa(
        y_test_t, y_pred_temp, y_prob_temp,
        'C. Re-entrenado temporal (2015-2021 → 2022-2025)')
    guardar_checkpoint({'y_pred': y_pred_temp, 'y_prob': y_prob_temp,
        'metricas': metricas_temp}, CP_TEMPORAL, 're-entrenamiento temporal DSSA')
    limpiar_memoria()

print('\nMétricas modelo re-entrenado temporalmente:')
display(pd.DataFrame([metricas_temp]))

# Cambio B → C
print('\nCambio de métricas: Modelo 03c→temporal (B) → Re-entrenado temporal (C)')
for metrica in ['AUC-PR', 'Bal_Acc', 'Recall', 'Specificity', 'Brier']:
    if metrica in metricas_temp and metrica in m_B:
        delta = metricas_temp[metrica] - m_B[metrica]
        if metrica == 'Brier':
            interp = '↓ mejora calibración' if delta < 0 else '↑ peor calibración'
        else:
            interp = '↑ mejora' if delta > 0 else '↓ sin mejora'
        print(f'  Δ{metrica:<15}: {delta:+.4f}  {interp}')
memoria_usada()

## 5. Tabla comparativa — tres escenarios

La comparativa DSSA tiene una interpretación diferente a la DCSA:
como no hay ruptura estructural, los cambios en métricas reflejan
degradación o mejora real del modelo, no artefactos distribucionales.

Los resultados se analizan en la celda siguiente.

In [ ]:
# Tabla comparativa final — tres escenarios
m_C = metricas_temp.copy()
m_C['Escenario'] = 'C. Re-entrenado\ntemporal'
m_A_plot = m_A.copy()
m_A_plot['Escenario'] = 'A. Test aleatorio\n(03c)'
m_B_plot = m_B.copy()
m_B_plot['Escenario'] = 'B. Modelo 03c\nsobre test temporal'

tabla_final = pd.DataFrame([m_A_plot, m_B_plot, m_C])
display(tabla_final.set_index('Escenario'))
tabla_final.to_csv(OUT_TAB / 'comparativa_temporal_DSSA.csv', index=False)

# Gráfico comparativo
metricas_plot  = ['AUC-PR', 'AUC-ROC', 'Recall', 'Specificity', 'Bal_Acc']
etiquetas_plot = ['AUC-PR\n(primaria)', 'AUC-ROC', 'Recall\nHosp.', 'Specificity\nNo Hosp.', 'Balanced\nAccuracy']
colores_esc    = [COLORS['neutral'], COLORS['test'], COLORS['primary']]
labels_esc     = ['A. Test aleatorio (03c)', 'B. Modelo 03c→temporal', 'C. Re-entrenado temporal']

fig, ax = plt.subplots(figsize=(13, 6))
x = np.arange(len(metricas_plot))
w = 0.26

for i, (row, label) in enumerate(zip([m_A_plot, m_B_plot, m_C], labels_esc)):
    vals   = [row.get(m, 0) for m in metricas_plot]
    offset = (i - 1) * w
    bars   = ax.bar(x + offset, vals, w,
        color=colores_esc[i], alpha=0.87, label=label, edgecolor='white')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2,
            bar.get_height() + 0.008,
            f'{val:.3f}',
            ha='center', va='bottom', fontsize=7, rotation=90,
            color=colores_esc[i])

# Prevalencia como piso de referencia para AUC-PR
ax.axhline(y_test_t.mean(), color='gray', linestyle=':',
    linewidth=1.5, alpha=0.7,
    label=f'Prevalencia test: {y_test_t.mean():.1%} (piso AUC-PR)')
ax.axhline(0.5, color='red', linestyle=':', linewidth=1.2, alpha=0.5,
    label='Bal. Acc = 0.5 (referencia azar)')

ax.set_xticks(x)
ax.set_xticklabels(etiquetas_plot, fontsize=10)
ax.set_ylabel('Puntuación')
ax.set_ylim(0, 1.18)
ax.set_title(
    'Comparativa de escenarios — DSSA\n'
    'Sin ruptura estructural: cambios reflejan degradación real',
    fontsize=12, fontweight='bold')
ax.legend(fontsize=8, loc='upper right', ncol=2)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
fig.savefig(OUT_FIG / 'temporal_dssa_comparativa.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

# Curvas ROC y Precision-Recall
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
if tiene_aleatorio:
    RocCurveDisplay.from_predictions(
        y_test_al, y_prob_al, name='A. Test aleatorio',
        color=COLORS['neutral'], ax=axes[0])
    PrecisionRecallDisplay.from_predictions(
        y_test_al, y_prob_al, name='A. Test aleatorio',
        color=COLORS['neutral'], ax=axes[1])
RocCurveDisplay.from_predictions(
    y_test_t, y_prob_03c, name='B. Modelo 03c→temporal',
    color=COLORS['test'], ax=axes[0])
RocCurveDisplay.from_predictions(
    y_test_t, y_prob_temp, name='C. Re-entrenado temporal',
    color=COLORS['primary'], ax=axes[0])
PrecisionRecallDisplay.from_predictions(
    y_test_t, y_prob_03c, name='B. Modelo 03c→temporal',
    color=COLORS['test'], ax=axes[1])
PrecisionRecallDisplay.from_predictions(
    y_test_t, y_prob_temp, name='C. Re-entrenado temporal',
    color=COLORS['primary'], ax=axes[1])
axes[0].plot([0,1],[0,1],'k--',alpha=0.4)
axes[0].set_title('A. Curvas ROC')
axes[0].grid(alpha=0.3)
axes[1].axhline(y_test_t.mean(), linestyle='--', color='gray', alpha=0.5,
    label=f'Prevalencia: {y_test_t.mean():.1%}')
axes[1].set_title('B. Curvas Precision-Recall (métrica primaria DSSA)')
axes[1].legend(fontsize=8)
axes[1].grid(alpha=0.3)
plt.suptitle('Robustez temporal — DSSA | Tres escenarios',
    fontsize=13, fontweight='bold')
plt.tight_layout()
fig.savefig(OUT_FIG / 'temporal_dssa_curvas.png',
    dpi=300, bbox_inches='tight', facecolor='white')
plt.show()

## 6. Conclusiones de la validación temporal — DSSA

### Contraste fundamental con DCSA

La validación temporal del DSSA confirma que los dos estratos
tienen naturalezas fundamentalmente diferentes:

| Aspecto | DCSA (03b) | DSSA (03d) |
|---------|-----------|----------|
| Tipo de drift | Metodológico — cambio protocolo INS | Epidemiológico — posible cambio serotipo |
| Cambio en tasa hosp. | +47 pp (40%→87%) | **~+0.3 pp (3.7%→4.0%)** |
| Artefacto en F1 temporal | Sí — F1 sube artificialmente | **No — cambios reflejan realidad** |
| Degradación en Bal. Acc | Severa (0.737→0.564) | Evaluar en resultados |
| Causa raíz | Formulario incompleto 2015-2021 | Variación epidemiológica real |

### Interpretación de los resultados

Los resultados numéricos están en las tablas anteriores.
Los patrones esperados y su significado:

**Si AUC-PR baja en el escenario B respecto a A:**
el modelo pierde capacidad discriminativa en el período reciente,
posiblemente porque el drift en ERUPCIONR y ARTRALGIA afecta
las reglas aprendidas sobre estos síntomas.

**Si AUC-PR se mantiene estable:**
el modelo generaliza razonablemente — la señal clínica en DSSA
es suficientemente robusta a los cambios epidemiológicos observados.

**Si el escenario C mejora sobre B:**
re-entrenar con datos del período antiguo aporta algo, pero con solo
~146 positivos en train y los años 2017–2020 vacíos,
el re-entrenamiento es muy ruidoso.

### Implicaciones para el despliegue

1. **La estabilidad de la prevalencia en DSSA es favorable:**
   a diferencia del DCSA, no hay ruptura que invalide el modelo por
   cambio en la distribución del target.
2. **El drift epidemiológico real requiere monitoreo:**
   si los serotipos circulantes siguen cambiando, la importancia
   de ERUPCIONR y ARTRALGIA variará — el modelo debería re-entrenarse
   periódicamente con datos recientes.
3. **El AUC-PR bajo en 03c es el problema principal:**
   más que la degradación temporal, el limitante es la señal clínica
   insuficiente en el formulario SIVIGILA para este estrato.
4. **Recomendación:** antes de desplegar cualquier modelo DSSA,
   enriquecer los features con variables de laboratorio o signos vitales
   cuantitativos — la validación temporal no cambia esta conclusión.

**Fin del módulo 03 — Modelado supervisado.**
**Siguiente:** `04_clustering/04_analisis_no_supervisado.ipynb`